# 01. Specialized Language Model for Low-Resource African Language (Twi)

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Corpus**: `ghana-nlp/abena-twi-corpus` (Hugging Face)  
**Objective**: Develop and evaluate a statistical n-gram language model for Akan/Twi using cloud streaming and sample scaling, comparing smoothing algorithms (MLE, Laplace, Lidstone, Interpolation, Kneser-Ney).

---
### Pipeline Overview
1. **Cloud Streaming with Scale Factor**: Stream `ghana-nlp/abena-twi-corpus` directly from Hugging Face Hub using `SCALE_FACTOR = 0.05` to prevent memory crashes.
2. **Corpus Curation**: Exclude archaic religious texts (e.g. Bible) in favor of balanced local news, culture, and contemporary text.
3. **Tokenization & Sentence Boundaries**: Prepend start token `<s>` and append `</s>`, isolating vocabulary strictly to the training split.
4. **N-Gram Model Training**: Unigram, Bigram, and Trigram count estimation.
5. **Smoothing Evaluation**: Resolving the zero-probability dilemma and comparing test perplexities.

In [ ]:
import sys
import random
from pathlib import Path

# Ensure repo root is on python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    load_twi_streaming_corpus,
    basic_tokenize,
    build_vocabulary,
    replace_oov_tokens,
)
from src.ngram import NGramLM
from src.viz import plot_ngram_frequency, plot_perplexity_comparison

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 1. Cloud Streaming & Sample Scaling
We stream lines directly from `ghana-nlp/abena-twi-corpus` with a sample scaling factor `SCALE_FACTOR = 0.05` (5%) to ensure memory efficiency in local and Google Colab environments.

In [ ]:
SCALE_FACTOR = 0.05
print(f"Pipeline scale factor set to: {SCALE_FACTOR * 100:.1f}%")

try:
    corpus_lines = load_twi_streaming_corpus(
        dataset_name="ghana-nlp/abena-twi-corpus",
        split="train",
        scale_factor=SCALE_FACTOR,
        seed=RANDOM_SEED,
        max_samples=2500,
    )
except Exception as e:
    print("Cloud streaming notice:", e)
    print("Falling back to representative curated Twi sentences for offline development...")
    corpus_lines = [
        "Me ma wo akye",
        "Wo ho te sen?",
        "Me ho ye, medaase",
        "Kofi reko sukuu wo Nkran",
        "Ama to aduane pa wo fie",
        "Yenko fie ntem efise osuo reto",
        "Okyena meba wo nkyen wo sukuu mu",
        "Abofra yi nim nyansa papaapa wo adesua mu",
        "Yebedi nkunim wo adesua mu daa",
        "Osuani no sua ade yiye",
        "Kofi ne Ama koo dwam to nnooma",
        "Yen nyinaa pe asomdwoe wo Ghana ha",
        "Wo din de sen?",
        "Me din de Kwabena",
        "Medaase papaapa wo mmoa no ho"
    ]

print(f"Loaded {len(corpus_lines)} sentences.")

## 2. Unicode Tokenization & Leak-Free Train/Test Split
We tokenize preserving Akan/Twi orthographic characters (`ɛ`, `ɔ`) and enforce strict split-first vocabulary construction to avoid test leakage.

In [ ]:
# Tokenize preserving African unicode glyphs
tokenized_data = [basic_tokenize(line) for line in corpus_lines if line.strip()]

# 80/20 train/test split
split_idx = int(0.8 * len(tokenized_data))
train_tokens = tokenized_data[:split_idx]
test_tokens = tokenized_data[split_idx:]

# Induce closed vocabulary STRICTLY from training set
vocab, freqs = build_vocabulary(train_tokens, min_freq=1)
train_clean = replace_oov_tokens(train_tokens, vocab)
test_clean = replace_oov_tokens(test_tokens, vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Training instances: {len(train_clean)}, Test instances: {len(test_clean)}")

## 3. Training Bigram & N-Gram Models with Smoothing
We train statistical models with prepended `<s>` start tokens to evaluate conditional probabilities: $P(w_1 \mid \text{<s>})$.

In [ ]:
# 1. Unigram with Laplace
unigram = NGramLM(n=1, smoothing="laplace").fit(train_clean, vocab=vocab)

# 2. Bigram with Laplace (Add-1)
bigram_laplace = NGramLM(n=2, smoothing="laplace", k=1.0).fit(train_clean, vocab=vocab)

# 3. Bigram with Lidstone (Add-0.1)
bigram_lidstone = NGramLM(n=2, smoothing="laplace", k=0.1).fit(train_clean, vocab=vocab)

# 4. Trigram with Linear Interpolation
trigram_interp = NGramLM(n=3, smoothing="interpolation").fit(train_clean, vocab=vocab)
trigram_interp.set_interpolation_weights([0.1, 0.3, 0.6])

# 5. Bigram with Interpolated Kneser-Ney (Continuation Probabilities)
bigram_kn = NGramLM(n=2, smoothing="kneser_ney").fit(train_clean, vocab=vocab)

print("All n-gram models trained successfully.")

## 4. Intrinsic Evaluation: Perplexity on Unseen Test Split

In [ ]:
models = {
    "Unigram (Laplace)": unigram,
    "Bigram (Laplace)": bigram_laplace,
    "Bigram (Add-0.1)": bigram_lidstone,
    "Trigram (Interpolation)": trigram_interp,
    "Bigram (Kneser-Ney)": bigram_kn,
}

results = {}
for name, model in models.items():
    ppl = model.perplexity(test_clean)
    results[name] = ppl
    print(f"{name:25s} -> Perplexity: {ppl:.2f}")

# Plot comparison
plot_perplexity_comparison(
    list(results.keys()),
    list(results.values()),
    title="Twi Language Model - Test Perplexity Benchmark",
    save_path=REPO_ROOT / "figures" / "ngram_perplexity_comparison.png",
)

## 5. Text Generation with Sampling Temperature

In [ ]:
print("--- Generated Twi Text Samples ---")
for name, model in models.items():
    sample = model.generate(max_length=10, temperature=0.7)
    print(f"[{name}]: {sample}")